# Experiment 8: Train a Neural Network using Keras/TensorFlow

**Objective:** To train a simple neural network model on a dataset using Keras/TensorFlow.

## Theory

The standard Keras workflow to train a model:
1. Prepare the dataset (features `X`, labels `y`).
2. Build the model architecture (`Sequential` + `Dense` layers).
3. Compile the model (optimizer, loss function, metrics).
4. Train the model (`model.fit`).
5. Observe how loss decreases and accuracy improves over epochs.

We use the classic **Iris flower dataset** (from scikit-learn) — a well-known multi-class classification problem — to classify flowers into 3 species from 4 measured features.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset  : Iris flowers ({X.shape[0]} samples, {X.shape[1]} features, 3 classes)")
print(f"Training : {X_train.shape[0]} samples")
print(f"Testing  : {X_test.shape[0]} samples")


Dataset  : Iris flowers (150 samples, 4 features, 3 classes)
Training : 120 samples
Testing  : 30 samples


## Build & Compile the Model

In [2]:
model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(10, activation="relu", name="hidden_layer_1"),
    layers.Dense(8, activation="relu", name="hidden_layer_2"),
    layers.Dense(3, activation="softmax", name="output_layer"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 8)              │            88 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165 (660.00 B)

 Trainable params: 165 (660.00 B)

 Non-trainable params: 0 (0.00 B)

## Train the Model

In [3]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=8,
    verbose=0,
)

print(f"{'Epoch':>6} | {'Train Loss':>11} | {'Train Acc':>10} | {'Val Loss':>10} | {'Val Acc':>8}")
print("-" * 58)
for i in range(0, 50, 10):
    print(f"{i + 1:>6} | {history.history['loss'][i]:>11.4f} | "
          f"{history.history['accuracy'][i]:>10.4f} | "
          f"{history.history['val_loss'][i]:>10.4f} | "
          f"{history.history['val_accuracy'][i]:>8.4f}")


 Epoch |  Train Loss |  Train Acc |   Val Loss |  Val Acc
----------------------------------------------------------
     1 |      0.9978 |     0.4417 |     1.0055 |   0.3667
    11 |      0.5153 |     0.8083 |     0.5337 |   0.7333
    21 |      0.3501 |     0.8917 |     0.3939 |   0.8333
    31 |      0.2473 |     0.9167 |     0.3004 |   0.8667
    41 |      0.1689 |     0.9667 |     0.2126 |   0.9333


## Final Evaluation

In [4]:
final_loss, final_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Final test accuracy: {final_acc * 100:.2f}%")
print(f"Final test loss    : {final_loss:.4f}")


Final test accuracy: 96.67%
Final test loss    : 0.1541


## Conclusion

The MLP trained successfully on the Iris dataset, achieving high test accuracy. Keras internally handled the forward pass, loss computation, backpropagation, and weight updates automatically for every epoch.